Create Fact Table

read silver data

In [0]:
df_silver = spark.sql("""
                      
                         SELECT * FROM PARQUET.`abfss://silver@carsalesdata.dfs.core.windows.net/carsales`
                      """
                      )
df_silver.display() 

In [0]:
df_dealer = spark.sql("select * from cars_catalog.gold.dim_dealer")
df_branch = spark.sql("select * from cars_catalog.gold.dim_branch")
df_model = spark.sql("select * from cars_catalog.gold.dim_model")
df_date = spark.sql("select * from cars_catalog.gold.dim_date")

df_branch.display()

In [0]:
df_fact = df_silver.join(df_branch,df_silver.Branch_ID == df_branch.Branch_ID, how ='left')\
    .join(df_dealer,df_silver.Dealer_ID == df_dealer.Dealer_ID, how ='left')\
    .join(df_model,df_silver.Model_ID == df_model.Model_ID, how ='left')\
    .join(df_date,df_silver.Date_ID == df_date.Date_ID, how ='left')\
    .select(df_silver.Revenue,df_silver.Units_Sold,df_branch.dim_branch_key,df_dealer.dim_dealer_key,df_model.dim_model_key,df_date.dim_date_key)

df_fact.display()

In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists('factsales'):
    deltatable = DeltaTable.forName(spark,'cars_catalog.gold.factsales')
    deltatable.alias('t').merge(df_fact.alias('s'),"t.dim_branch_key = s.dim_branch_key and t.dim_dealer_key = s.dim_dealer_key and t.dim_model_key = s.dim_model_key and t.dim_date_key = s.dim_date_key")\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()
else:
    df_fact.write.format('delta')\
        .mode('Overwrite')\
        .option("path","abfss://gold@carsalesdata.dfs.core.windows.net/fact_sales")\
        .saveAsTable('cars_catalog.gold.factsales')
        

In [0]:
%sql
select * from cars_catalog.gold.factsales;